# Task 3 
> Study the various disambiguation algorithms implemented in https://github.com/alvations/pywsd.
> Show that you can run the program on a simple sentence containing a target word to disambiguate and
> output for various disambiguation algorithms (Original Lesk, adapted/extended Lesk, simple Lesk
> extended with hypernym and hyponyms, cosine Lesk, path-semantic similarity, Information contentsemantic similarity, highest > Lemma count.

[pywsd](https://github.com/alvations/pywsd) is a Python library that collects many word sense
disambiguation algorithms based on WordNet. In this task we run them on a few example sentences and compare
their answers.

## The Lesk family
All four algorithms choose the sense whose **signature** (a set of words describing the sense) has the most words in common with the sentence. They differ in **what goes into the signature** and **how the match is measured**.

- **Original Lesk**: the signature is only the **definition**, and words are compared as they are (no lowercasing, no stopword removal).
- **Simple Lesk**: a richer signature, with definition, **examples**, **synonyms**, **hypernyms** and **hyponyms**. Words are lowercased, lemmatized and stopwords are removed.
- **Adapted/Extended Lesk**: like Simple Lesk, plus **holonyms**, **meronyms** and similar senses, so the signature is even larger.
- **Cosine Lesk**: same signature as Simple Lesk, but the score is the **cosine similarity** instead of a count, so a shared word counts more in a short signature than in a long one.


In [1]:
import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)  
nltk.download("punkt_tab", quiet=True)                  

from pywsd._wordnet import wn  
from pywsd.lesk import original_lesk, simple_lesk, adapted_lesk, cosine_lesk
from pywsd.utils import lemmatize
from pywsd.similarity import max_similarity
from pywsd.baseline import max_lemma_count
from pywsd.lesk import signatures, simple_signatures
from pywsd.utils import lemmatize_sentence

To task the algorithms we have chosen three sentences:
1. "I have been prescribed two important drugs today during my visit to clinic", the same sentence as task 1.
2. "I went to the bank to deposit my money", same experiment as the pywsd README, to check that everything works.
3. "The left ventricle of the heart pumps blood to the body", a medical sentence with two medical senses in WordNet (heart vs brain)

We first look at the WordNet senses of each target word, because the algorithms can only choose among them.

In [2]:
# The three test sentences: (sentence, target word)
SENTENCES = [
    ("I have been prescribed two important drugs today during my visit to clinic", "drug"),
    ("I went to the bank to deposit my money", "bank"),
    ("The left ventricle of the heart pumps blood to the body", "ventricle"),
]

# The correct sense of each target word (chosen by us, by reading the definitions)
CORRECT = {
    "drug": "oewn-03252323-n",       # a substance that is used as a medicine or narcotic
    "bank": "oewn-08437235-n",       # a financial institution that accepts deposits...
    "ventricle": "oewn-05399106-n",  # a chamber of the heart...
}

In [3]:
def mark(sense, target):
    """Return a tick if the sense is the correct one."""
    return "✓" if sense.id == CORRECT[target] else "✗"

def lesk_ranking(algorithm, sentence, target):
    """Ranking (score, sense, words in common) of all senses, computed as pywsd does."""
    if algorithm == "original":
        sigs = signatures(lemmatize(target), original_lesk=True)   
        context = sentence.split()                                  
    elif algorithm == "simple":
        sigs = simple_signatures(lemmatize(target))                
        context = lemmatize_sentence(sentence)
    else:  # adapted
        sigs = signatures(lemmatize(target), adapted=True)        
        context = lemmatize_sentence(sentence)
    ranking = [(len(set(sig) & set(context)), sense, sorted(set(sig) & set(context)))
               for sense, sig in sigs.items()]
    return sorted(ranking, key=lambda x: x[0], reverse=True)      


LESK_ALGORITHMS = {
    "Original Lesk": "original",
    "Simple Lesk": "simple",
    "Adapted/Extended Lesk": "adapted",
}

for sentence, target in SENTENCES:
    correct_def = wn.synset(CORRECT[target]).definition()
    print(f"\n{'=' * 90}\n{target.upper()}   (correct sense: {correct_def[:55]}...)\n{'=' * 90}")

    # Count-based Lesk: score = number of words in common
    for name, algorithm in LESK_ALGORITHMS.items():
        ranking = lesk_ranking(algorithm, sentence, target)
        chosen = ranking[0][1]
        print(f"\n{name}  ->  {mark(chosen, target)}  {chosen.definition()[:60]}")
        for rank, (score, sense, common) in enumerate(ranking[:3], start=1):
            arrow = "<- chosen" if rank == 1 else ""
            print(f"     {rank}. score {score:3}  {mark(sense, target)}  {sense.definition()[:45]:45}  common: {common} {arrow}")
        position = [s.id for _, s, _ in ranking].index(CORRECT[target]) + 1
        if position > 3:
            print(f"     ... correct sense is at position {position} of {len(ranking)}")

    # Cosine Lesk: score = cosine similarity (no list of common words)
    ranking = cosine_lesk(sentence, target, nbest=True)
    chosen = ranking[0][1]
    print(f"\nCosine Lesk  ->  {mark(chosen, target)}  {chosen.definition()[:60]}")
    for rank, (score, sense) in enumerate(ranking[:3], start=1):
        arrow = "<- chosen" if rank == 1 else ""
        print(f"     {rank}. score {score:.3f}  {mark(sense, target)}  {sense.definition()[:45]:45} {arrow}")
    position = [s.id for _, s in ranking].index(CORRECT[target]) + 1
    if position > 3:
        print(f"     ... correct sense is at position {position} of {len(ranking)}")

Download [##############################] (12912118/12912118 bytes) Completeg
Read [##############################] (1298444/1298444) .xmll
Added oewn:2024 (Open English Wordnet) (1456931/1456931) Examplesonsionssurs




DRUG   (correct sense: a substance that is used as a medicine or narcotic...)

Original Lesk  ->  ✗  administer a drug to
     1. score   1  ✗  administer a drug to                           common: ['to'] <- chosen
     2. score   1  ✗  use recreational drugs                         common: ['drugs'] 
     3. score   0  ✓  a substance that is used as a medicine or nar  common: [] 

Simple Lesk  ->  ✓  a substance that is used as a medicine or narcotic
     1. score   1  ✓  a substance that is used as a medicine or nar  common: ['drug'] <- chosen
     2. score   1  ✗  administer a drug to                           common: ['drug'] 
     3. score   1  ✗  use recreational drugs                         common: ['drug'] 

Adapted/Extended Lesk  ->  ✓  a substance that is used as a medicine or narcotic
     1. score   1  ✓  a substance that is used as a medicine or nar  common: ['drug'] <- chosen
     2. score   1  ✗  administer a drug to                           common: ['drug'] 
     3.

**Observations:**
- Every algorithm correctly identifies the meaning of *ventricle*, because *heart*, *blood* and *pump* appear
  both in the sentence and in the definition of the correct sense. For Original Lesk, however, half of the
  shared words are stopwords (*of, the, to*).
- Original Lesk fails on *drug* because it does not lemmatize and does not remove stopwords. The sentence
  contains *drugs* (plural), so the singular *drug* in "administer a **drug** to" does not match; this sense
  gets 1 point only because of the stopword *to*. The sense "use recreational **drugs**" also gets 1 point,
  while the correct sense gets 0. The tie is broken in favour of the first sense in the list, which is wrong.
- Simple and Adapted Lesk get *drug* right, but **only by a tie**: after lemmatization (*drugs* → *drug*)
  and stopword removal, the only word in common for all three senses is *drug* itself, so all of them score 1
  and the first sense wins. The context gives no real clue.
- For *bank*, three algorithms choose the correct sense ("a financial institution..."), while Cosine Lesk
  chooses "a container for keeping money". Both senses are related to money, which makes them hard to
  separate. Original Lesk is also right only by a tie (three senses with 2 points, partly due to *the*).
  Simple and Adapted Lesk, instead, win clearly thanks to *deposit* and *money* (3 points vs 2).

Now let's have a look at the similarity-based methods implemented in pywsd. Unlike the Lesk family, they do
not count words in common: they use the **is-a hierarchy** of WordNet (hypernyms). For each sense of the
target word, the algorithm computes how similar it is to the senses of the other words in the sentence,
sums these values and chooses the sense with the highest total.

The difference between the methods is **how the similarity between two senses is measured**:

- **Path similarity** – based on the length of the shortest path between the two senses in the hierarchy:
  the shorter the path, the more similar they are.
- **Wu-Palmer** – based on the depth of their closest common ancestor: two senses that meet deep in the
  hierarchy (in a specific concept) are more similar than two senses that meet near the top.
- **Leacock-Chodorow** – like path similarity, but the path length is scaled by the total depth of the
  hierarchy (only for senses with the same part of speech).
- **Resnik** – based on the **information content** (IC) of the common ancestor, i.e. how specific and rare
  it is in a corpus: a more specific common ancestor means more similar senses.
- **Jiang-Conrath** and **Lin** – variants of Resnik that also take into account the IC of the two senses
  themselves, not only of their common ancestor.

Finally, **Max lemma count** is a baseline that ignores the sentence and always returns the most frequent sense.

In [4]:
SIMILARITY_OPTIONS = {
    "Path similarity": "path",
    "Wu-Palmer": "wup",
    "Leacock-Chodorow": "lch",
    "Resnik (IC)": "res",
    "Jiang-Conrath (IC)": "jcn",
    "Lin (IC)": "lin",
}

In [5]:
for sentence, target in SENTENCES:
    correct_def = wn.synset(CORRECT[target]).definition()
    print(f"\n{'=' * 90}\n{target.upper()}   (correct sense: {correct_def[:55]}...)\n{'=' * 90}")

    for name, option in SIMILARITY_OPTIONS.items():
        ranking = max_similarity(sentence, target, option=option, best=False)
        chosen = ranking[0][1]                       # the sense returned by pywsd
        print(f"\n{name}  ->  {mark(chosen, target)}  {chosen.definition()[:60]}")
        for rank, (score, sense) in enumerate(ranking[:3], start=1):
            arrow = "<- chosen" if rank == 1 else ""
            print(f"     {rank}. score {score:6.2f}  {mark(sense, target)}  {sense.definition()[:50]:50} {arrow}")
        # where is the correct sense in the ranking?
        position = [s.id for _, s in ranking].index(CORRECT[target]) + 1
        if position > 3:
            print(f"     ... correct sense is at position {position} of {len(ranking)}")

    sense = max_lemma_count(target)
    print(f"\nMax lemma count (no scores, no context)  ->  {mark(sense, target)}  {sense.definition()[:60]}")


DRUG   (correct sense: a substance that is used as a medicine or narcotic...)

Path similarity  ->  ✓  a substance that is used as a medicine or narcotic
     1. score   1.86  ✓  a substance that is used as a medicine or narcotic <- chosen
     2. score   1.64  ✗  use recreational drugs                             
     3. score   1.42  ✗  administer a drug to                               

Wu-Palmer  ->  ✓  a substance that is used as a medicine or narcotic
     1. score   3.61  ✓  a substance that is used as a medicine or narcotic <- chosen
     2. score   2.20  ✗  use recreational drugs                             
     3. score   2.09  ✗  administer a drug to                               

Leacock-Chodorow  ->  ✓  a substance that is used as a medicine or narcotic
     1. score  14.57  ✓  a substance that is used as a medicine or narcotic <- chosen
     2. score   9.92  ✗  use recreational drugs                             
     3. score   8.82  ✗  administer a drug to          

**Observations:**
- For *drug*, Path, Wu-Palmer, Leacock-Chodorow and Lin choose the correct sense with a clear margin
  (e.g. Wu-Palmer 3.61 vs 2.20). Jiang-Conrath is also right, but only by a tie: every sense gets an infinite
  score, so the first sense in the list wins.
- Resnik fails on *drug* because of the bug in pywsd: the ranking is sorted in ascending order, so the sense
  with the **lowest** score (13.55) is returned. Note that even with the bug fixed Resnik would choose the
  wrong sense, since the highest score (15.36) belongs to "administer a drug to".
- For *bank* every method fails, and the correct sense is only at position 7–10 out of 18. The scores of the
  top senses are very close (e.g. Wu-Palmer 2.86 vs 2.73), so with many senses the choice becomes almost
  random, and some methods even choose a verb ("put into a bank account").
- For *ventricle* every method chooses the brain ventricle. The margins are tiny (e.g. Wu-Palmer 3.82 vs
  3.65), and for Resnik, Jiang-Conrath and Lin the two senses have exactly the same score, so the first sense
  wins. These measures only use the **is-a** hierarchy of WordNet, so they cannot see that the ventricle is
  **part of** the heart.
- Jiang-Conrath always gives infinite scores because the target word also appears in the context: comparing a
  sense with itself gives a distance of 0, and its formula is 1 / distance. In practice it always returns the
  first sense, like Max lemma count.
- Max lemma count always returns the first WordNet sense (Open English WordNet has no sense frequency
  counts): it is right for *drug* but wrong for *bank* ("sloping land") and *ventricle* (brain).